In [ ]:
import os
import json
from argparse import Namespace as _NS
from typing import List, Tuple
from pathlib import Path

import numpy as np
import torch
import matplotlib.pyplot as plt

# Reuse components from the training script
from normalized_dirty_awan import UniversalReconstructionDataset, AWANCleanNet

# Configuration
checkpoint_path = 'checkpoints/awan_dirty_random_crop_192p_200ch_8b_psnl_20250119_143022.pth'  # Update this path
log_file_path = 'logs/training_history_20250119_143022.json'  # Update this path for training metrics visualization
catalog_path = ''  # Set on target machine

def extract_paths(list_file: str, catalog_path: str) -> List[Tuple[str, str, str]]:
    paths: List[Tuple[str, str, str]] = []
    with open(list_file, 'r') as f:
        for line in f:
            if not line.strip():
                continue
            dat = catalog_path + line.strip()
            hdr = catalog_path + line.split('.dat')[0] + '.hdr'
            ident = line.split('/results/')[0].split('/')[-1]
            png = catalog_path + line.split('/results/')[0] + '/' + ident + '.png'
            paths.append((hdr, dat, png))
    return paths


In [ ]:
# Load checkpoint and extract args
ckpt = torch.load(checkpoint_path, map_location='cpu', weights_only=False)

# Try to read args from checkpoint (new format)
if 'args' in ckpt:
    print("Loading args from checkpoint metadata...")
    args_dict = ckpt['args']
    args = _NS(**args_dict)
    print(f"Loaded args: render={args.render}, channels={args.channels}, blocks={args.blocks}, use_psnl={not args.no_psnl}")
else:
    # Fallback to filename parsing (backward compatibility)
    print("Args not found in checkpoint, parsing filename...")
    filename = Path(checkpoint_path).stem
    parts = filename.split('_')
    
    args = _NS(
        train_path='train_list.txt',
        waves='fx10_wavelengths_224.npy',
        css='css_fx10_204.npy',
        catalog_path=catalog_path,
        ckpt_path=checkpoint_path,
        channels=200,
        blocks=8,
        use_psnl=True,
        num_samples=4,
        tau=10.0,
        render=False,
        random_crop=False,
    )
    
    # Parse from filename
    if len(parts) > 0:
        args.render = parts[0].split('awan_')[-1] == 'clean' if 'clean' in parts[0] else False
    if len(parts) > 1:
        args.random_crop = parts[1] == 'random'
    if len(parts) > 2:
        # Extract channels
        for p in parts:
            if 'ch' in p:
                args.channels = int(p.split('ch')[0].split('_')[-1])
    if len(parts) > 2:
        # Extract blocks
        for p in parts:
            if 'b' in p and 'ch' not in p:
                args.blocks = int(p.split('b')[0].split('_')[-1])
    if len(parts) > 2:
        # Extract psnl
        args.use_psnl = 'nopsnl' not in filename
    if len(parts) > 2:
        # Extract tau
        for p in parts:
            if 't' in p and p != 'nopsnl':
                try:
                    args.tau = float(p.split('t')[0].split('_')[-1])
                except:
                    pass

print(f"Final args: render={args.render}, channels={args.channels}, blocks={args.blocks}, use_psnl={args.use_psnl}")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


In [ ]:
# Load wavelengths and CSS
waves = np.load(args.waves).astype('float32')
css = np.load(args.css).astype('float32')

# Build model consistent with training
model = AWANCleanNet(bands_out=css.shape[0], channels=args.channels, n_blocks=args.blocks, use_psnl=args.use_psnl)
model.load_state_dict(ckpt['model'], strict=False)
model.eval().to(device)

# Dataset that returns real RGB (from PNG) and GT spectrum
items = extract_paths(args.train_path, args.catalog_path)
ds = UniversalReconstructionDataset(items, waves, css, patch_size=None, augment=False, render=args.render)

print(f"Model loaded. Checkpoint info:")
print(f"  Epoch: {ckpt.get('ep', 'N/A')}")
print(f"  Best RMSE: {ckpt.get('best_rmse', 'N/A'):.4f}" if 'best_rmse' in ckpt else "  Best RMSE: N/A")
print(f"  Timestamp: {ckpt.get('timestamp', 'N/A')}")


In [ ]:
def show_mean(i):
    """Plot mean spectrum comparison for sample i."""
    sample = ds[i]
    rgb = sample['rgb'].unsqueeze(0).to(device)  # (1,3,H,W)
    gt_spec = sample['hsi'].squeeze().cpu().numpy().mean((1, 2))  # (Bands,)
    pred_spec = model(rgb).squeeze().detach().cpu().numpy().mean((1, 2))  # (Bands,)

    plt.figure(figsize=(7, 4))
    plt.plot(waves, gt_spec, label='GT mean spectrum', linewidth=2)
    plt.plot(waves, pred_spec, label='Pred mean spectrum', linewidth=2)
    plt.xlabel('Wavelength (nm)')
    plt.ylabel('Reflectance')
    plt.ylim(0, 1.2)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
def show_pics(gt_spec, pred_spec, rgb, offset=0):
    """Show channel-wise comparison images."""
    fig, ax = plt.subplots(figsize=(14, 8), nrows=2, ncols=4)
    for j in range(len(ax[0])):
        ax[0][j].imshow(gt_spec.cpu().numpy()[j + offset], cmap='viridis')
        ax[0][j].set_title(f'GT Channel {waves[j + offset]:.2f} nm')
        ax[0][j].axis('off')
    for j in range(len(ax[1])):
        ax[1][j].imshow(pred_spec.cpu().numpy()[j + offset], cmap='viridis')
        ax[1][j].set_title(f'Pred Channel {waves[j + offset]:.2f} nm')
        ax[1][j].axis('off')
    
    fig.text(0.5, 0.92, "GT images", ha='center', fontsize=14, fontweight='bold')
    fig.text(0.5, 0.46, "Pred images", ha='center', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()


In [ ]:
def per_pixel_spectral_cov_corr(gt, pred, bands_dim=0, unbiased=True, eps=1e-12):
    """Compute per-pixel spectral covariance and correlation."""
    # Ensure float tensors on same device
    gt = gt.to(dtype=torch.float32)
    pred = pred.to(device=gt.device, dtype=torch.float32)

    # Reorder to (Bands, H, W)
    if bands_dim == -1:
        gt = gt.permute(2, 0, 1)
        pred = pred.permute(2, 0, 1)
    elif bands_dim != 0:
        raise ValueError("bands_dim must be 0 or -1")

    B, H, W = gt.shape
    if B < 2 and unbiased:
        unbiased = False

    # Per-pixel means over bands
    mu_x = gt.mean(dim=0)      # (H, W)
    mu_y = pred.mean(dim=0)    # (H, W)

    # Centered
    xm = gt - mu_x             # broadcasts to (B, H, W)
    ym = pred - mu_y

    denom = (B - 1) if unbiased else B

    # Covariance per pixel across bands
    cov_map = (xm * ym).sum(dim=0) / max(denom, 1)  # (H, W)

    # Variances per pixel
    var_x = (xm * xm).sum(dim=0) / max(denom, 1)
    var_y = (ym * ym).sum(dim=0) / max(denom, 1)

    # Correlation per pixel
    corr_map = cov_map / (var_x.sqrt() * var_y.sqrt() + eps)
    corr_map = corr_map.clamp(-1.0, 1.0)

    return cov_map, corr_map

def plot_heatmap(arr2d, title=None):
    """Plot 2D heatmap."""
    arr2d_cpu = arr2d.detach().float().cpu()
    plt.figure(figsize=(6, 5))
    im = plt.imshow(arr2d_cpu, origin='upper', cmap='viridis')
    plt.colorbar(im, fraction=0.046, pad=0.04)
    if title:
        plt.title(title)
    plt.tight_layout()
    plt.show()


In [ ]:
def plot_training_metrics(log_file_path):
    """Visualize training metrics from log file."""
    if not os.path.exists(log_file_path):
        print(f"Log file not found: {log_file_path}")
        return
    
    with open(log_file_path, 'r') as f:
        log_data = json.load(f)
    
    history = log_data.get('history', [])
    if not history:
        print("No training history found in log file")
        return
    
    epochs = [entry['epoch'] for entry in history]
    train_loss = [entry['train']['loss'] for entry in history]
    val_rmse = [entry['val']['rmse'] for entry in history]
    val_mrae = [entry['val']['mrae'] for entry in history]
    
    # Plot training metrics
    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    
    # Training loss
    axes[0, 0].plot(epochs, train_loss, 'b-', linewidth=2, label='Train Loss')
    axes[0, 0].set_xlabel('Epoch')
    axes[0, 0].set_ylabel('Loss')
    axes[0, 0].set_title('Training Loss')
    axes[0, 0].grid(True, alpha=0.3)
    axes[0, 0].legend()
    
    # Validation RMSE
    axes[0, 1].plot(epochs, val_rmse, 'r-', linewidth=2, label='Val RMSE')
    axes[0, 1].set_xlabel('Epoch')
    axes[0, 1].set_ylabel('RMSE')
    axes[0, 1].set_title('Validation RMSE')
    axes[0, 1].grid(True, alpha=0.3)
    axes[0, 1].legend()
    
    # Validation MRAE
    axes[1, 0].plot(epochs, val_mrae, 'g-', linewidth=2, label='Val MRAE')
    axes[1, 0].set_xlabel('Epoch')
    axes[1, 0].set_ylabel('MRAE')
    axes[1, 0].set_title('Validation MRAE')
    axes[1, 0].grid(True, alpha=0.3)
    axes[1, 0].legend()
    
    # Combined plot
    ax2 = axes[1, 0].twinx()
    ax2.plot(epochs, val_rmse, 'r--', linewidth=2, alpha=0.7, label='Val RMSE')
    ax2.set_ylabel('RMSE', color='r')
    ax2.tick_params(axis='y', labelcolor='r')
    
    # Loss components (if available)
    if 'l_hsi' in history[0]['train']:
        train_l_hsi = [entry['train']['l_hsi'] for entry in history]
        train_l_rgb = [entry['train']['l_rgb'] for entry in history]
        axes[1, 1].plot(epochs, train_l_hsi, 'b-', linewidth=2, label='L_HSI')
        axes[1, 1].plot(epochs, train_l_rgb, 'g-', linewidth=2, label='L_RGB')
        axes[1, 1].set_xlabel('Epoch')
        axes[1, 1].set_ylabel('Loss Component')
        axes[1, 1].set_title('Training Loss Components (Clean Mode)')
        axes[1, 1].grid(True, alpha=0.3)
        axes[1, 1].legend()
    elif 'mrae' in history[0]['train']:
        train_mrae = [entry['train']['mrae'] for entry in history]
        axes[1, 1].plot(epochs, train_mrae, 'm-', linewidth=2, label='Train MRAE')
        axes[1, 1].set_xlabel('Epoch')
        axes[1, 1].set_ylabel('MRAE')
        axes[1, 1].set_title('Training MRAE (Dirty Mode)')
        axes[1, 1].grid(True, alpha=0.3)
        axes[1, 1].legend()
    else:
        axes[1, 1].text(0.5, 0.5, 'No loss components available', 
                        ha='center', va='center', transform=axes[1, 1].transAxes)
        axes[1, 1].set_title('Loss Components')
    
    plt.tight_layout()
    plt.show()
    
    # Print summary
    print(f"\nTraining Summary:")
    print(f"  Total epochs: {len(history)}")
    print(f"  Best RMSE: {min(val_rmse):.4f} at epoch {epochs[val_rmse.index(min(val_rmse))]}")
    print(f"  Best MRAE: {min(val_mrae):.4f} at epoch {epochs[val_mrae.index(min(val_mrae))]}")
    print(f"  Final RMSE: {val_rmse[-1]:.4f}")
    print(f"  Final MRAE: {val_mrae[-1]:.4f}")

# Visualize training metrics if log file exists
if os.path.exists(log_file_path):
    plot_training_metrics(log_file_path)
else:
    print(f"Log file not found at {log_file_path}. Skipping training metrics visualization.")


In [ ]:
# Example usage: visualize a sample
import random

i = 5  # Sample index
sample = ds[i]
rgb = sample['rgb'].unsqueeze(0).to(device)  # (1,3,H,W)
gt_spec = sample['hsi'].squeeze()  # (Bands, H, W)
pred_spec = model(rgb).squeeze().detach()  # (Bands, H, W)

# Show mean spectrum
show_mean(i)

# Show channel-wise images
show_pics(gt_spec, pred_spec, rgb, offset=100)

# Show per-pixel spectral correlation
cov_map, corr_map = per_pixel_spectral_cov_corr(gt_spec, pred_spec, bands_dim=0, unbiased=True)
plot_heatmap(corr_map, "Per-pixel spectral correlation (Pearson r)")


In [ ]:
# Clean up GPU memory
torch.cuda.empty_cache()
